# MultiUN Translation Samples Demo

Compare 300 cached samples across Arabic, Chinese, English, French, Russian, and Spanish: 50 per language, each with 2,000 Llama-3.2-3B content tokens.

Source: [MultiUN](https://huggingface.co/datasets/Helsinki-NLP/multiun), training splits of `ar-en`, `en-zh`, `en-fr`, `en-ru`, and `en-es`. Translations are matched through shared English text; token limits can produce different endpoints.

Light cleanup normalized whitespace and control characters, decoded HTML entities, and removed leading paragraph numbers when the same number appeared in only some translations.

Run all cells from `MultiUN-translations` to compare cached samples side by side. Diagnostics are computed by `Generate-multiun-translation-spectral-diagnostics.ipynb`.

In [1]:
from pathlib import Path
from typing import Any, Dict, Optional

import html as _html
import json

from IPython.display import HTML, clear_output, display

PREVIEW_TOKENS = 500
DATA_DIR = Path("multiun_language_samples")
COMBINED_JSONL_PATH = DATA_DIR / "multiun_language_2000_token_windows.jsonl"
LANGUAGE_NAMES = {
    "ar": "Arabic",
    "zh": "Chinese",
    "en": "English",
    "fr": "French",
    "ru": "Russian",
    "es": "Spanish"
}
LANGUAGES = list(LANGUAGE_NAMES)

if not COMBINED_JSONL_PATH.is_file():
    raise FileNotFoundError(
        f"Cached sample file not found: {COMBINED_JSONL_PATH.name}. "
        "Run this notebook from MultiUN-translations, or set DATA_DIR to the sample folder. "
        "If the file is missing, restore the combined JSONL distributed with this demo."
    )

with COMBINED_JSONL_PATH.open(encoding="utf-8") as sample_file:
    browser_records = [json.loads(line) for line in sample_file if line.strip()]
if not browser_records:
    raise ValueError(f"The cached sample file is empty: {COMBINED_JSONL_PATH}")

browser_records_by_run_language = {}
required_fields = {"sample_id", "run_id", "language", "text", "token_ids", "num_tokens", "preview_text_500_tokens"}
for record in browser_records:
    missing_fields = required_fields.difference(record)
    if missing_fields:
        raise ValueError(
            f"A cached sample is missing fields: {sorted(missing_fields)}. "
            "Restore the combined JSONL distributed with this demo."
        )
    if not isinstance(record["text"], str) or not isinstance(record["preview_text_500_tokens"], str):
        raise ValueError(f"Cached display text must be a string for {record['sample_id']}.")
    if len(record["token_ids"]) != record["num_tokens"]:
        raise ValueError(f"Saved token counts disagree for {record['sample_id']}.")
    key = (int(record["run_id"]), str(record["language"]))
    if key in browser_records_by_run_language:
        raise ValueError(f"Duplicate sample for run and language: {key}")
    if key[1] not in LANGUAGE_NAMES:
        raise ValueError(f"Unrecognized cached language: {key[1]}")
    browser_records_by_run_language[key] = record

run_options = sorted({key[0] for key in browser_records_by_run_language})
language_options = [language for language in LANGUAGES if any(key[1] == language for key in browser_records_by_run_language)]
if language_options != LANGUAGES or any(
    (run_id, language) not in browser_records_by_run_language
    for run_id in run_options for language in language_options
):
    raise ValueError("Each cached sample must be available in all six languages.")

print(f"Loaded {len(browser_records):,} cached samples: {len(run_options)} per language.")

Loaded 300 cached samples: 50 per language.


Select a sample and two languages to compare the first 500 tokens or the full 2,000-token text. Both views load directly from cached samples. Interactive controls require `ipywidgets`; static previews below also work without widgets.

In [2]:
def preview_text(record: Dict[str, Any], preview_tokens: Optional[int] = PREVIEW_TOKENS) -> str:
    if preview_tokens is None:
        text = record["text"]
    elif preview_tokens == PREVIEW_TOKENS:
        text = record["preview_text_500_tokens"]
    else:
        raise ValueError("Choose the cached first 500 tokens or the full sample.")
    return text.strip()


def render_comparison(run_id: int, left_language: str, right_language: str, preview_tokens: Optional[int]) -> HTML:
    left_record = browser_records_by_run_language[(int(run_id), str(left_language))]
    right_record = browser_records_by_run_language[(int(run_id), str(right_language))]
    left_title = f"{left_language} - {LANGUAGE_NAMES.get(left_language, left_language)}"
    right_title = f"{right_language} - {LANGUAGE_NAMES.get(right_language, right_language)}"
    left_text = _html.escape(preview_text(left_record, preview_tokens))
    right_text = _html.escape(preview_text(right_record, preview_tokens))

    style = """
    <style>
      .mun-wrap {
        border: 1px solid #d5d7dc;
        border-radius: 8px;
        overflow: hidden;
        font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
        background: #ffffff;
        width: 100%;
      }
      .mun-head, .mun-body {
        display: grid;
        grid-template-columns: minmax(0, 1fr) minmax(0, 1fr);
      }
      .mun-head {
        background: #f5f6f8;
        border-bottom: 1px solid #d5d7dc;
        font-weight: 650;
        color: #1f2937;
      }
      .mun-head > div, .mun-text {
        padding: 12px 14px;
      }
      .mun-head > div:first-child, .mun-text:first-child {
        border-right: 1px solid #eceef2;
      }
      .mun-text {
        white-space: pre-wrap;
        overflow-wrap: anywhere;
        line-height: 1.55;
        color: #111827;
        max-height: 520px;
        overflow: auto;
      }
    </style>
    """
    return HTML(
        style
        + "<div class='mun-wrap'>"
        + "<div class='mun-head'>"
        + f"<div>{_html.escape(left_title)}</div>"
        + f"<div>{_html.escape(right_title)}</div>"
        + "</div>"
        + "<div class='mun-body'>"
        + f"<div class='mun-text'>{left_text}</div>"
        + f"<div class='mun-text'>{right_text}</div>"
        + "</div>"
        + "</div>"
    )


try:
    import ipywidgets as widgets
except ImportError:
    widgets = None
    print("Interactive controls are unavailable. The static previews below still work.")

if widgets is not None:
    left_language_dropdown = widgets.Dropdown(
        options=[(f"{language} - {LANGUAGE_NAMES.get(language, language)}", language) for language in language_options],
        value="en" if "en" in language_options else language_options[0],
        description="Left",
        layout=widgets.Layout(width="240px"),
    )
    right_language_dropdown = widgets.Dropdown(
        options=[(f"{language} - {LANGUAGE_NAMES.get(language, language)}", language) for language in language_options],
        value="fr" if "fr" in language_options else language_options[min(1, len(language_options) - 1)],
        description="Right",
        layout=widgets.Layout(width="240px"),
    )
    sample_dropdown = widgets.Dropdown(
        options=[(f"Sample {run_id:02d}", int(run_id)) for run_id in run_options],
        value=int(run_options[0]),
        description="Sample",
        layout=widgets.Layout(width="180px"),
    )
    view_dropdown = widgets.Dropdown(
        options=[("First 500", PREVIEW_TOKENS), ("Full", None)],
        value=PREVIEW_TOKENS,
        description="View",
        layout=widgets.Layout(width="180px"),
    )
    output = widgets.Output(layout=widgets.Layout(width="100%"))


    def render_selected_sample(*_):
        with output:
            clear_output(wait=True)
            display(render_comparison(sample_dropdown.value, left_language_dropdown.value, right_language_dropdown.value, view_dropdown.value))


    left_language_dropdown.observe(render_selected_sample, names="value")
    right_language_dropdown.observe(render_selected_sample, names="value")
    sample_dropdown.observe(render_selected_sample, names="value")
    view_dropdown.observe(render_selected_sample, names="value")

    render_selected_sample()
    sample_browser = widgets.VBox([
        widgets.HBox([left_language_dropdown, right_language_dropdown, sample_dropdown, view_dropdown]),
        output,
    ])
    display(sample_browser)

Static previews show all six languages in pairs for the same sample. Change `PREVIEW_RUN_ID` to choose another sample or set `STATIC_PREVIEW_TOKENS = None` to show full text.

In [3]:
PREVIEW_RUN_ID = run_options[0]
STATIC_PREVIEW_TOKENS = PREVIEW_TOKENS

for left_language, right_language in [("en", "fr"), ("ar", "zh"), ("ru", "es")]:
    display(render_comparison(PREVIEW_RUN_ID, left_language, right_language, STATIC_PREVIEW_TOKENS))